# 第三級 · 訓練自己的金屬零件模型（Colab T4）

案例資料：**SteelDS**（輸送帶金屬廢料，YOLO 格式、CC BY 4.0，Zenodo `10.5281/zenodo.20271102`）。
流程：下載子集 → Colab T4 微調 yolo11n → 驗證 mAP → 在測試影像上看結果 → 匯出 NCNN 給邊緣。

> 訓練需要 GPU，請先：執行階段 → 變更執行階段類型 → **T4 GPU**。

In [ ]:
# 1) 確認 GPU（訓練一定要 GPU）
import torch
assert torch.cuda.is_available(), '訓練需 GPU：執行階段→變更執行階段類型→T4 GPU'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2) 安裝
!pip -q install ultralytics
import ultralytics; print(ultralytics.__version__)

## 取得資料集（擇一）
- **方式 A（推薦課堂・快）**：Roboflow「Bolts」螺絲/螺帽/墊圈，小、幾分鐘訓完（需免費 Roboflow API key）。
- **方式 B（SteelDS 輸送帶金屬・較大）**：從 Zenodo 下載子集，無需金鑰。a4 最小（約 4.9GB）。

In [ ]:
# 方式 A：Roboflow Bolts（需在 roboflow.com 取免費 API key）
# !pip -q install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_KEY')
# ds = rf.workspace('bolts').project('bolts-final').version(1).download('yolov8')
# DATA = ds.location + '/data.yaml'
# print('DATA =', DATA)

In [ ]:
# 方式 B：SteelDS 子集（無金鑰）。a4 最小；要更大可改 a1
import os, glob, yaml
SUBSET = 'a4'
if not os.path.exists(SUBSET):
    !wget -q -O {SUBSET}.zip "https://zenodo.org/records/20271102/files/{SUBSET}.zip?download=1"
    !unzip -q {SUBSET}.zip -d .
# 找出實際的 train/test images 目錄（不同打包可能多一層）
trains = glob.glob(f'**/{SUBSET}/train/images', recursive=True) + glob.glob('**/train/images', recursive=True)
root = os.path.dirname(os.path.dirname(trains[0]))
print('資料根目錄:', root); print('train 影像數:', len(glob.glob(root+'/train/images/*')))
DATA = 'steelds.yaml'
yaml.safe_dump({'path': os.path.abspath(root), 'train':'train/images', 'val':'test/images',
  'nc':3, 'names':{0:'Background',1:'Steel',2:'Copper'}}, open(DATA,'w'))
print('已寫 data.yaml ->', DATA)

In [ ]:
# 3) Colab T4 微調（yolo11n，batch 調小以符合 T4 16GB）
from ultralytics import YOLO
m = YOLO('yolo11n.pt')
m.train(data=DATA, epochs=30, imgsz=640, batch=8, name='metal', exist_ok=True)
print('best 權重: runs/detect/metal/weights/best.pt')

In [ ]:
# 4) 驗證 mAP
best = YOLO('runs/detect/metal/weights/best.pt')
mt = best.val(data=DATA)
print('mAP50:', round(mt.box.map50,3), '| mAP50-95:', round(mt.box.map,3))

In [ ]:
# 5) 在測試影像上看結果（存標註圖；若資料集附影片可改 track 影片）
import glob, os
root = os.path.dirname(os.path.dirname(glob.glob('**/test/images', recursive=True)[0]))
imgs = sorted(glob.glob(root+'/test/images/*'))[:6]
best.predict(imgs, save=True, project='result', name='metal', exist_ok=True, conf=0.3)
print('結果圖在 result/metal/，可下載或顯示')
from IPython.display import Image as IPImage, display
for p in sorted(glob.glob('result/metal/*'))[:3]: display(IPImage(p, width=520))

In [ ]:
# 6) 匯出 NCNN 給邊緣（第二級 RPi 用）
best.export(format='ncnn')
print('已匯出 NCNN：runs/detect/metal/weights/best_ncnn_model')